# Network search: BFS, Dijkstra and A*

**Algorithms:** `bfs_distances` and `shortest_path` from `zarr_vectors_tools.algorithms`

Both read all of the level's edges, including those between chunks, into an
in-memory adjacency, then search it.

1. Generate a graph whose edges carry their Euclidean length
2. Write it with `length` as an edge attribute
3. `bfs_distances`: hop counts and the predecessor tree from one vertex
4. `shortest_path` with `weight="length"`: Dijkstra
5. `shortest_path` with a straight-line heuristic: A*

In [1]:
import os
import tempfile

import numpy as np
from zarr_vectors.types.graphs import read_graph, write_graph
from zarr_vectors_tools.algorithms import bfs_distances, shortest_path

_tmpdir = tempfile.mkdtemp(prefix="zv_examples_")
STORE = os.path.join(_tmpdir, "vessels.zv")
print("Store path:", STORE)

Store path: /tmp/zv_examples_2xctkwyu/vessels.zv


## 1 · Generate a weighted graph

4,000 random vertices in a 600 µm cube, each joined to its five nearest
neighbours; each edge's `length` is the distance between its ends.

In [2]:
from scipy.spatial import KDTree

rng = np.random.default_rng(11)
n_nodes = 4_000
positions = rng.uniform(0, 600, (n_nodes, 3)).astype(np.float32)   # µm

_, nbrs = KDTree(positions).query(positions, k=6)   # the first neighbour is the vertex itself
src = np.repeat(np.arange(n_nodes), 5)
dst = nbrs[:, 1:].ravel()
edges = np.unique(np.sort(np.column_stack([src, dst]), axis=1), axis=0).astype(np.int32)
edge_len = np.linalg.norm(positions[edges[:, 0]] - positions[edges[:, 1]], axis=1).astype(np.float32)

print(f"vertices : {n_nodes:,}")
print(f"edges    : {len(edges):,}")
print(f"length   : min {edge_len.min():.2f}, max {edge_len.max():.2f}, mean {edge_len.mean():.2f} µm")

vertices : 4,000
edges    : 12,157
length   : min 1.36, max 76.81, mean 34.20 µm


## 2 · Write the graph

In [3]:
summary = write_graph(
    STORE,
    positions=positions,
    edges=edges,
    chunk_shape=(150.0, 150.0, 150.0),
    bin_shape=(37.5, 37.5, 37.5),
    kind="graph",
    link_attributes={"length": edge_len},
)
print(summary)

{'node_count': 4000, 'edge_count': 12157, 'chunk_count': 64, 'intra_edge_count': 9306, 'cross_edge_count': 2851, 'object_count': 1, 'kind': 'graph'}


## 3 · Breadth-first search

Vertices are numbered in store order: row *i* of `read_graph(...)["positions"]`
is vertex *i*. The source is the vertex nearest one corner of the cube and the
target the vertex nearest the opposite corner. `distances` holds hop counts,
`-1` where a vertex is not reached; `predecessors[v]` is the lowest-numbered
neighbour of `v` one hop nearer the source.

In [4]:
P = read_graph(STORE)["positions"]
source = int(np.argmin(np.linalg.norm(P - [0, 0, 0], axis=1)))
target = int(np.argmin(np.linalg.norm(P - [600, 600, 600], axis=1)))
print(f"source = {source}  at {P[source].round(1)}")
print(f"target = {target}  at {P[target].round(1)}")

bfs = bfs_distances(STORE, source)
dist, preds = bfs["distances"], bfs["predecessors"]
print(f"\nreached     : {(dist >= 0).sum():,} of {len(dist):,} vertices")
print(f"max hops    : {dist.max()}")
print(f"target hops : {dist[target]}")

path_bfs = [target]
while path_bfs[-1] != source:
    path_bfs.append(int(preds[path_bfs[-1]]))
path_bfs.reverse()
bfs_len = np.linalg.norm(np.diff(P[path_bfs], axis=0), axis=1).sum()
print(f"BFS path    : {len(path_bfs) - 1} hops, {bfs_len:.1f} µm")

source = 27  at [ 8.7  6.1 53.1]
target = 3962  at [559.8 579.6 557.6]

reached     : 4,000 of 4,000 vertices
max hops    : 31
target hops : 31
BFS path    : 31 hops, 1319.9 µm


## 4 · Dijkstra

`weight="length"` makes each edge cost its stored `length`, so the result is
the geometrically shortest path. It usually takes more hops than the BFS path
and is shorter in µm.

In [5]:
sp = shortest_path(STORE, source, target, weight="length")
euclid = np.linalg.norm(np.diff(P[sp["path"]], axis=0), axis=1).sum()
print(f"hops    : {len(sp['path']) - 1}")
print(f"cost    : {sp['cost']:.2f} µm")
print(f"visited : {sp['visited']:,} vertices")
print(f"sum of edge lengths along the path: {euclid:.2f} µm")
assert abs(sp["cost"] - euclid) < 1e-2

hops    : 35
cost    : 1192.91 µm
visited : 4,000 vertices
sum of edge lengths along the path: 1192.91 µm


## 5 · A*

`heuristic(v)` must not overestimate the remaining cost. The straight-line
distance to the target never does when edges cost their length, so A* returns
a path of the same cost as Dijkstra while taking fewer vertices off the
queue.

In [6]:
target_pos = P[target]

def straight_line(v: int) -> float:
    return float(np.linalg.norm(P[v] - target_pos))

sp_astar = shortest_path(STORE, source, target, weight="length", heuristic=straight_line)
print(f"hops    : {len(sp_astar['path']) - 1}")
print(f"cost    : {sp_astar['cost']:.2f} µm  (Dijkstra {sp['cost']:.2f} µm)")
print(f"visited : {sp_astar['visited']:,} vertices  (Dijkstra {sp['visited']:,})")
assert abs(sp_astar["cost"] - sp["cost"]) < 1e-3

hops    : 35
cost    : 1192.91 µm  (Dijkstra 1192.91 µm)
visited : 1,605 vertices  (Dijkstra 4,000)


## Summary

| Step | API |
|------|-----|
| BFS | `bfs_distances(store, source, max_distance=None)` → `distances`, `predecessors` |
| Dijkstra | `shortest_path(store, source, target, weight="length")` → `path`, `cost`, `visited` |
| A* | `shortest_path(store, source, target, weight="length", heuristic=fn)` |

`weight=None` costs 1 per edge. If the store has no edge attribute of the
given name, every edge weighs 1, without error. `shortest_path` returns
`{'path': [], 'cost': inf}` when the target cannot be reached.